# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `mlcroissant` library organizes tabular (and other) data into **record sets**. Each record set contains **fields** (columns), both identified by their `@id`. Use these `@id`s to extract and analyze data.

In [ ]:
# List all record sets and their fields with @id and name
print('Available record sets:')
record_sets = list(dataset.record_sets())
for rset in record_sets:
    print(f"- RecordSet name: {rset.name} (@id: {rset.id})")
    if hasattr(rset, 'fields'):
        for field in rset.fields:
            print(f"  - Field: {field.name} (@id: {field.id}) - type: {field.data_type}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Each record set and field is referenced by its `@id` as above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rset.id for rset in dataset.record_sets()]

# For this notebook, we'll print IDs and then choose the primary table to work with
print('RecordSet IDs:', record_set_ids)

# Typically, for a mass spectrometry dataset, there is one main table (e.g., for proteins/peptides)
# Let's pick the first record set as the main one
main_record_set_id = record_set_ids[0]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} rows for RecordSet {record_set_id}")

# Show available columns for the main record set and first few rows
print('Columns:', dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize a numeric field, remove outliers or group the data.

We use field and record set `@id`s as in all previous cells.

In [ ]:
# First, list numeric fields in the main record set to select one for EDA
main_rset = None
for rset in dataset.record_sets():
    if rset.id == main_record_set_id:
        main_rset = rset
        break
# Get numeric fields
numeric_fields = [field.id for field in main_rset.fields if field.data_type in ['Number', 'Float', 'Integer']]
print('Numeric fields:', numeric_fields)

# Choose a numeric field (using the @id of the first numeric field found)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Analyzing field: {numeric_field_id}")
else:
    raise ValueError("No numeric fields found.")

df = dataframes[main_record_set_id].copy()

# Filter for values above given threshold (e.g., 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field, if available
categorical_fields = [field.id for field in main_rset.fields if field.data_type in ['Text', 'String']]
group_field_id = categorical_fields[0] if categorical_fields else None
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll plot the distribution of the selected numeric field and show group means if available.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of normalized numeric field
plt.figure(figsize=(8,5))
plt.hist(filtered_df[f"{numeric_field_id}_normalized"], bins=30, alpha=0.7)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Frequency")
plt.show()

# If grouping by a categorical field was possible, plot average per group
if group_field_id and group_field_id in filtered_df.columns:
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    group_means.plot(kind='bar', figsize=(10,4), title=f'Average {numeric_field_id} by {group_field_id}')
    plt.ylabel(f"Average {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

Key steps included:
- Enumerating record sets and fields using their `@id`s, per Croissant best practice.
- Extracting one or more record sets to pandas DataFrames for analysis.
- Filtering and normalizing data on numeric fields.
- (If available) Grouping and visualizing values by categorical fields.

For deeper analysis, refer to the full Croissant schema and dataset documentation, and extend this notebook by referencing additional fields or performing domain-specific analyses.